# Model Optimization

**Module:** 05 — LLM Fundamentals

Quantization, pruning, distillation, KV cache, and Flash Attention—speed/memory levers.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain INT8/INT4/GPTQ-style quantization tradeoffs
- Distinguish pruning and distillation goals
- Describe KV cache and Flash Attention benefits
- Pick optimizations for a latency/memory budget


## Quantization — INT8 / INT4 / GPTQ

**Definition.** Store/compute weights (and sometimes activations) in lower precision to save memory/bandwidth.

**Why it matters.** Often the highest-leverage deploy optimization for LLMs.

**How it works.** Post-training quant (GPTQ/AWQ) or quant-aware training; validate quality.

**Intuition.** Compress the encyclopedia without losing too many words.

**Common pitfalls.**
- Not evaluating task quality after quant
- Mixing incompatible quant formats

**When to use.** Edge/GPU memory limits; cost reduction.

| Format | Footprint | Risk |
|--------|-----------|------|
| FP16 | High | Baseline |
| INT8 | ~1/2 | Usually small drop |
| INT4/GPTQ | ~1/4 | Task-sensitive |


In [ ]:
# Demo 1 — memory scale
P=7e9
for name, b in [("fp16",2),("int8",1),("int4",0.5)]:
    print(name, P*b/1e9, "GB weights")


In [ ]:
# Demo 2 — error sketch
import numpy as np
w = np.linspace(-1,1,8)
q = np.round(w*2)/2  # crude
print(list(zip(w.round(2), q)))


In [ ]:
# Demo 3 — GPTQ idea
print("use calibration data to minimize quantization error layer-wise")


### Try it yourself — Quantization — INT8 / INT4 / GPTQ

1. List two metrics you'd compare before/after INT4.


## Pruning

**Definition.** Remove weights/neurons/heads with small contribution to reduce compute/size.

**Why it matters.** Can shrink models; structured pruning helps real speedups.

**How it works.** Magnitude/structured pruning + recovery fine-tune.

**Intuition.** Trim dead branches of the tree.

**Common pitfalls.**
- Unstructured sparsity without sparse kernels
- Over-pruning attention heads

**When to use.** When you control serving stacks that exploit sparsity.


In [ ]:
# Demo 1 — magnitude prune
import numpy as np
w = np.array([0.01, -1.2, 0.02, 0.8])
thr = 0.05
print(np.where(np.abs(w)<thr, 0, w))


In [ ]:
# Demo 2 — structured vs unstructured
print({"unstructured": "zeros inside dense mats", "structured": "drop heads/channels"})


### Try it yourself — Pruning

1. Why might 50% unstructured zeros not yield 2× speed?


## Distillation

**Definition.** Train a smaller student to mimic a larger teacher's outputs/logits/behaviors.

**Why it matters.** Deploy cheaper models that keep much of teacher quality on a task distribution.

**How it works.** KL on logits / match responses; sometimes rationales.

**Intuition.** Apprentice copies the master on the syllabus you care about.

**Common pitfalls.**
- Distilling away safety behavior
- Mismatched student capacity

**When to use.** High-QPS task-specific serving.


In [ ]:
# Demo 1 — KL sketch
import numpy as np
t = np.array([0.7,0.2,0.1]); s=np.array([0.5,0.3,0.2])
kl = np.sum(t*np.log((t+1e-9)/(s+1e-9)))
print(kl)


In [ ]:
# Demo 2 — response distillation dataset
print({"teacher_answer": "...", "student_train_target": "..."})


### Try it yourself — Distillation

1. Propose a student task distribution for a support bot.


## KV Cache

**Definition.** Store per-layer key/value tensors for past tokens so decode doesn't recompute them.

**Why it matters.** Makes autoregressive decoding feasible at length.

**How it works.** Allocate cache proportional to layers×heads×dim×tokens×batch; manage eviction for long sessions.

**Intuition.** Notes from earlier paragraphs so you don't reread the whole book each word.

**Common pitfalls.**
- OOM from long chats
- Forgetting cache invalidation when prompt changes

**When to use.** All modern LLM servers.


In [ ]:
# Demo 1 — cache bytes
def kv_mb(tokens, layers=32, kv_heads=8, hd=128, bytes_per=2):
    return tokens*layers*2*kv_heads*hd*bytes_per / 1e6
print(kv_mb(8192), "MB")


In [ ]:
# Demo 2 — without cache cost
prompt_T, new_T = 1000, 50
print("naive recompute token-steps", sum(prompt_T+i for i in range(new_T)))


### Try it yourself — KV Cache

1. Estimate KV memory for 32k context on a 70B-class config (rough).


## Flash Attention

**Definition.** IO-aware attention kernels that reduce memory reads/writes, speeding attention and lowering memory footprint.

**Why it matters.** Enables longer contexts and faster training/inference on GPUs.

**How it works.** Tiling softmax attention to keep data in SRAM; exact or carefully approximated variants.

**Intuition.** Smarter memory traffic, not a new math definition of attention.

**Common pitfalls.**
- Assuming Flash Attention changes model quality materially
- Hardware/kernel incompatibilities

**When to use.** Whenever supported on your stack—usually free lunch.


In [ ]:
# Demo 1 — memory traffic intuition
print({"materialize_NxN": "bad for long T", "flash_tiled": "keeps chunks in fast memory"})


In [ ]:
# Demo 2 — complexity reminder
T=8192
print("attn score entries", T*T)


### Try it yourself — Flash Attention

1. In one sentence: what bottleneck does Flash Attention primarily attack?


## Glossary

- **GPTQ**: Post-training weight quantization method
- **KV cache**: Cached K/V for decode


### Workshop drill — Model Optimization (1)

Restate each section heading as a single exam-ready sentence.


In [ ]:
# Workshop drill 1 — Model Optimization
headings = ['Quantization — INT8 / INT4 / GPTQ', 'Pruning', 'Distillation', 'KV Cache', 'Flash Attention']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Model Optimization (2)

Change one hyperparameter/assumption in a demo and predict the effect before running.


In [ ]:
# Workshop drill 2 — Model Optimization
print('prediction: ...')
print('observation: ...')
print('delta: ...')


### Workshop drill — Model Optimization (3)

List production risks (cost, latency, safety, quality) for this topic.


In [ ]:
# Workshop drill 3 — Model Optimization
for r in ['cost','latency','safety','quality']:
    print(f'{r}:')


## Summary & Key Takeaways

- Quantization is usually the first deploy lever
- Distillation helps task-specific cheap serving
- KV cache is mandatory for decode; watch memory
- Flash Attention improves attention IO efficiency

### Practice

Pick a model size and propose an optimization stack for a 16GB GPU.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
